In [5]:
import pandas as pd

t3 = pd.read_csv("../data/processed/Expenditure/expenditure_avg_spending_per_person.csv")
t4 = pd.read_csv("../data/processed/Expenditure/expenditure_total_spending_by_state.csv")
t5 = pd.read_csv("../data/processed/Expenditure/expenditure_avg_spending_by_state.csv")
t28 = pd.read_csv("../data/processed/Expenditure/expenditure_spending_by_area.csv")
t29 = pd.read_csv("../data/processed/Expenditure/expenditure_hospital_funding_source.csv")
t30 = pd.read_csv("../data/processed/Expenditure/expenditure_public_hospital_funding_source.csv")
tA3 = pd.read_csv("../data/processed/Expenditure/expenditure_area_funding_matrix.csv")

In [6]:
t3.head()

,year,spending_current,spending_constant,nominal_change_pct,real_growth_pct
0,2013–14,6608.756533,8509.998967,NaN,NaN
1,2014–15,6801.533849,8615.419165,2.916998,1.238780
2,2015–16,7057.752875,8773.923652,3.767077,1.839777
3,2016–17,7379.930619,8972.751459,4.564877,2.266122
4,2017–18,7494.462830,8953.393931,1.551942,-0.215737


In [7]:
t4.head()

,year,state,spending
0,2013–14,nsw,62177.189085
1,2014–15,nsw,64133.401368
2,2015–16,nsw,65046.507008
3,2016–17,nsw,65866.657831
4,2017–18,nsw,67822.626965


In [8]:
tA3.head()

,Area of expenditure,funding_source,spending
0,Hospitals,DVA,1153
1,Public hospital services,DVA,445
2,Private hospitals,DVA,708
3,Primary health care,DVA,860
4,Unreferred medical services,DVA,134


In [9]:
years = pd.concat([
    t3["year"],
    t4["year"],
    t5["year"],
    t28["Year"],
    t29["Year"],
    t30["Year"]
]).drop_duplicates().sort_values()

In [10]:
dim_year = pd.DataFrame({"year": years}).reset_index(drop=True)
dim_year["year_id"] = dim_year.index + 1

dim_year = dim_year[["year_id", "year"]]

dim_year

,year_id,year
0,1,2013–14
1,2,2014–15
2,3,2015–16
3,4,2016–17
4,5,2017–18
5,6,2018–19
6,7,2019–20
7,8,2020–21
8,9,2021–22
9,10,2022–23


In [14]:
states = pd.concat([
    t4["state"],
    t5["state"]
]).str.upper().drop_duplicates().sort_values()

In [15]:
dim_state = pd.DataFrame({"state": states}).reset_index(drop=True)

dim_state["state_id"] = dim_state.index + 1

dim_state = dim_state[["state_id", "state"]]

dim_state

,state_id,state
0,1,ACT
1,2,NSW
2,3,NT
3,4,QLD
4,5,SA
5,6,TAS
6,7,VIC
7,8,WA


In [16]:
areas = pd.concat([
    t28["expenditure_area"],
    tA3["Area of expenditure"]
]).drop_duplicates().sort_values()

In [17]:
dim_expenditure_area = pd.DataFrame({
    "expenditure_area": areas
}).reset_index(drop=True)

dim_expenditure_area["expenditure_area_id"] = dim_expenditure_area.index + 1

dim_expenditure_area = dim_expenditure_area[
    ["expenditure_area_id", "expenditure_area"]
]

dim_expenditure_area

,expenditure_area_id,expenditure_area
0,1,Administration
1,2,Aids and appliances
2,3,All other medications
3,4,Benefit-paid pharmaceuticals
4,5,Community health and other
5,6,Dental services
6,7,Other health practitioners
7,8,Patient transport services
8,9,Private hospitals
9,10,Public health


In [18]:
dim_expenditure_area = dim_expenditure_area[
    ~dim_expenditure_area["expenditure_area"].str.contains("Total", case=False)
].reset_index(drop=True)

In [19]:
dim_expenditure_area["expenditure_area_id"] = dim_expenditure_area.index + 1

In [20]:
dim_expenditure_area

,expenditure_area_id,expenditure_area
0,1,Administration
1,2,Aids and appliances
2,3,All other medications
3,4,Benefit-paid pharmaceuticals
4,5,Community health and other
5,6,Dental services
6,7,Other health practitioners
7,8,Patient transport services
8,9,Private hospitals
9,10,Public health


In [21]:
funding_sources = pd.concat([
    t29["funding_source"],
    t30["funding_source"],
    tA3["funding_source"]
]).drop_duplicates().sort_values()

In [22]:
dim_funding_source = pd.DataFrame({
    "funding_source": funding_sources
}).reset_index(drop=True)

dim_funding_source["funding_source_id"] = dim_funding_source.index + 1

dim_funding_source = dim_funding_source[
    ["funding_source_id", "funding_source"]
]

dim_funding_source

,funding_source_id,funding_source
0,1,Australian Government
1,2,DVA
2,3,HIF
3,4,Health and other
4,5,Individuals
5,6,Non-government
6,7,Other
7,8,Premium rebates
8,9,State and local
9,10,State and territory government


In [25]:
t4["state"] = t4["state"].str.upper()

In [26]:
fact_spending_by_state = (
    t4
    .merge(dim_state, on="state")
    .merge(dim_year, on="year")
)

In [27]:
fact_spending_by_state = fact_spending_by_state[
    ["state_id", "year_id", "spending"]
]

fact_spending_by_state.head()

,state_id,year_id,spending
0,2,1,62177.189085
1,2,2,64133.401368
2,2,3,65046.507008
3,2,4,65866.657831
4,2,5,67822.626965


In [28]:
fact_spending_by_state.shape

(88, 3)

In [31]:
print(tA3.columns)

Index(['Area of expenditure', 'funding_source', 'spending'], dtype='object')


In [32]:
tA3 = tA3.rename(columns={
    "Area of expenditure": "expenditure_area"
})

In [33]:
tA3.head()

,expenditure_area,funding_source,spending
0,Hospitals,DVA,1153
1,Public hospital services,DVA,445
2,Private hospitals,DVA,708
3,Primary health care,DVA,860
4,Unreferred medical services,DVA,134


In [39]:
tA3 = tA3[~tA3["expenditure_area"].isin([
    "Total health expenditure",
    "Total recurrent expenditure"
])]

In [40]:
fact_expenditure = (
    tA3
    .merge(dim_expenditure_area, on="expenditure_area")
    .merge(dim_funding_source, on="funding_source")
)

fact_expenditure = fact_expenditure[
    ["expenditure_area_id", "funding_source_id", "spending"]
]

In [41]:
fact_expenditure.shape

(133, 3)

In [46]:
import os
os.getcwd()

'C:\\AU-Healthcare\\notebooks'

In [47]:
os.makedirs("../data/modelled/Expenditure", exist_ok=True)

In [48]:
dim_year.to_csv("../data/modelled/Expenditure/dim_year.csv", index=False)
dim_state.to_csv("../data/modelled/Expenditure/dim_state.csv", index=False)
dim_expenditure_area.to_csv("../data/modelled/Expenditure/dim_expenditure_area.csv", index=False)
dim_funding_source.to_csv("../data/modelled/Expenditure/dim_funding_source.csv", index=False)

fact_spending_by_state.to_csv("../data/modelled/Expenditure/fact_spending_by_state.csv", index=False)
fact_expenditure.to_csv("../data/modelled/Expenditure/fact_expenditure.csv", index=False)